In [1]:
import pandas as pd

In [ ]:
train = pd.read_csv("data/train.csv")

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10517 entries, 0 to 10516
Columns: 1078 entries, LogP to mZagreb2
dtypes: float64(1078)
memory usage: 86.5 MB


In [4]:
X = train.drop(columns='LogP').reset_index(drop=True)
y = train['LogP']

In [6]:
print(X.shape)
print(y.shape)

(10517, 1077)
(10517,)


---

<h1>BaseLine Model(no feature selection)</h1>

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

linreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linreg.fit(X, y)


Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())])

In [9]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
import numpy as np


def objective_rf(trial):
    # 하이퍼파라미터 샘플링
    n_estimators = trial.suggest_int("n_estimators", 100, 600)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring="neg_mean_squared_error")

    return scores.mean()  # negative MSE (Optuna는 maximize 하므로 ok)


In [ ]:
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=50)

print("Best params:", study_rf.best_params)
print("Best score:", study_rf.best_value)


# [I 2025-12-06 19:04:57,500] Trial 0 finished with value: -2.998287340983373 and parameters: {'n_estimators': 508, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 1}. Best is trial 0 with value: -2.998287340983373.
#[I 2025-12-06 19:13:32,574] Trial 1 finished with value: -3.0145041796582213 and parameters: {'n_estimators': 157, 'max_depth': 6, 'min_samples_split': 3, 'min_samples_leaf': 3}. Best is trial 0 with value: -2.998287340983373.

[I 2025-12-06 17:48:09,808] A new study created in memory with name: no-name-6039aa02-2855-4b7b-9397-f6af570e52a6
[I 2025-12-06 19:04:57,500] Trial 0 finished with value: -2.998287340983373 and parameters: {'n_estimators': 508, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 1}. Best is trial 0 with value: -2.998287340983373.
[I 2025-12-06 19:13:32,574] Trial 1 finished with value: -3.0145041796582213 and parameters: {'n_estimators': 157, 'max_depth': 6, 'min_samples_split': 3, 'min_samples_leaf': 3}. Best is trial 0 with value: -2.998287340983373.


In [ ]:
best_rf_params = study_rf.best_params

rf = RandomForestRegressor(
    **best_rf_params,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)


In [ ]:
from lightgbm import LGBMRegressor

def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", -1, 20),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0)
    }

    model = LGBMRegressor(**params, random_state=42)

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring="neg_mean_squared_error")

    return scores.mean()


In [ ]:
study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=50)

print("Best params:", study_lgbm.best_params)


In [ ]:
best_lgbm_params = study_lgbm.best_params

lgbm = LGBMRegressor(
    **best_lgbm_params,
    random_state=42
)

lgbm.fit(X, y)
